In [ ]:
# 🔧 Setup: Run this cell first!
# Check GPU availability and install dependencies

import torch
import sys

# Check GPU
if torch.cuda.is_available():
    device = torch.device('cuda')
    print(f"✅ GPU available: {torch.cuda.get_device_name(0)}")
    print(f"   Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")
else:
    device = torch.device('cpu')
    print("⚠️ No GPU detected. Some cells may run slowly.")
    print("   Go to Runtime → Change runtime type → GPU")

print(f"\n📦 Python {sys.version.split()[0]}")
print(f"🔥 PyTorch {torch.__version__}")

# Set random seeds for reproducibility
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print(f"🎲 Random seed set to {SEED}")

%matplotlib inline

# Case Study: AML Transaction Screening with Prompt Design Principles
## Implementation Notebook

*Meridian Compliance — Building an LLM-powered compliance screening pipeline*
*Estimated time: 75 minutes*

## 1. Environment Setup and Dataset Loading

In this notebook, you will build an LLM-powered anti-money laundering (AML) transaction screening pipeline for Meridian Compliance, a Series B fintech serving 85 mid-market banks. The current rule-based system flags 42,000 transactions daily with a 97.1% false positive rate. Your mission: reduce false positives by 60% while maintaining 95%+ suspicious activity detection using prompt design techniques — system prompts, few-shot learning, chain-of-thought reasoning, structured output, and prompt chaining.

In [ ]:
# Install dependencies
!pip install anthropic numpy pandas matplotlib scikit-learn sentence-transformers -q

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (
    precision_recall_curve, confusion_matrix,
    classification_report, precision_score, recall_score
)

# We will use Claude as our LLM
from anthropic import Anthropic

# Initialize the client
# In Colab: set your API key in the secrets panel or environment
import os
client = Anthropic(api_key=os.environ.get("ANTHROPIC_API_KEY", "your-key-here"))
MODEL = "claude-sonnet-4-20250514"

print("Setup complete.")

## 2. Synthetic AML Transaction Dataset

We generate a synthetic dataset modeled on real AML typologies published by FinCEN and FATF. Real SAR data is confidential under 31 USC 5318(g)(2), but the statistical properties — amount distributions, country risk profiles, narrative patterns — are drawn from publicly available typology guides.

In [ ]:
# AML typologies based on FinCEN published patterns
AML_TYPOLOGIES = {
    "structuring": "Breaking large transactions into smaller amounts below reporting thresholds",
    "layering_shell_companies": "Moving funds through chains of shell companies in offshore jurisdictions",
    "trade_based": "Over- or under-invoicing goods to move value across borders",
    "funnel_accounts": "Using domestic accounts to aggregate and move illicit funds for foreign actors",
    "cash_intensive": "Using cash-heavy businesses to commingle illicit and legitimate funds",
    "rapid_movement": "Depositing and immediately wiring funds with no apparent business purpose",
    "nominee_accounts": "Using third parties to open and transact on accounts to obscure beneficial ownership",
    "digital_currency": "Converting between fiat and cryptocurrency to obscure transaction trails"
}

HIGH_RISK_COUNTRIES = ["VG", "KY", "PA", "BZ", "VU", "WS", "MH", "SC"]
MEDIUM_RISK_COUNTRIES = ["HK", "SG", "AE", "CH", "LU", "MT", "CY", "LI"]
LOW_RISK_COUNTRIES = ["US", "GB", "CA", "DE", "FR", "JP", "AU", "NL"]

np.random.seed(42)

def generate_benign_transaction(idx):
    """Generate a realistic benign transaction narrative."""
    templates = [
        "Domestic wire transfer from {company} to {vendor} for invoice #{inv}. "
        "Purpose: payment for {service}. Recurring monthly payment, consistent with "
        "account history. Amount: ${amount:,.2f}.",

        "ACH payment from {company} employee payroll account to {count} employees. "
        "Regular bi-weekly payroll processing. Amount: ${amount:,.2f}. "
        "Consistent with prior payroll cycles.",

        "International wire from {company} ({country1}) to {vendor} ({country2}). "
        "Purpose: quarterly licensing fee per contract dated {date}. "
        "Amount: ${amount:,.2f}. Transaction matches expected payment schedule.",

        "Check deposit from {name} into personal savings account. "
        "Source: tax refund from IRS. Amount: ${amount:,.2f}. "
        "Annual pattern consistent with prior years.",

        "Wire transfer from {company} to {vendor} for commercial real estate "
        "lease payment. Monthly recurring. Amount: ${amount:,.2f}. "
        "Matches signed lease agreement on file."
    ]

    companies = ["Acme Industries", "Bright Solutions LLC", "CoreTech Inc",
                 "Delta Manufacturing", "EastPoint Services", "FairView Holdings"]
    vendors = ["Office Depot", "AWS Cloud Services", "United Leasing Corp",
               "National Insurance Co", "Consolidated Logistics"]
    services = ["IT consulting services Q3", "annual software licenses",
                "warehouse lease Q4", "equipment maintenance contract",
                "professional development training"]
    names = ["John M. Patterson", "Sarah K. Williams", "Robert Chen",
             "Maria L. Garcia", "David Thompson"]

    template = np.random.choice(templates)
    amount = np.random.lognormal(mean=9.5, sigma=1.2)  # Median ~$13K
    amount = min(amount, 500000)

    country1 = np.random.choice(LOW_RISK_COUNTRIES)
    country2 = np.random.choice(LOW_RISK_COUNTRIES)

    narrative = template.format(
        company=np.random.choice(companies),
        vendor=np.random.choice(vendors),
        service=np.random.choice(services),
        name=np.random.choice(names),
        inv=np.random.randint(10000, 99999),
        count=np.random.randint(15, 200),
        date=f"20{np.random.randint(20,24)}-{np.random.randint(1,13):02d}-01",
        amount=amount,
        country1=country1,
        country2=country2
    )

    return {
        "transaction_id": f"TXN-2024-{idx:07d}",
        "narrative": narrative,
        "amount": round(amount, 2),
        "currency": "USD",
        "originator_country": country1,
        "beneficiary_country": country2,
        "transaction_type": np.random.choice(["wire", "ACH", "check"], p=[0.5, 0.3, 0.2]),
        "originator_entity_type": np.random.choice(["corporation", "individual"], p=[0.7, 0.3]),
        "customer_risk_rating": np.random.choice(["low", "medium"], p=[0.8, 0.2]),
        "rule_triggers": [],
        "label": 0,
        "typology": "none"
    }


def generate_suspicious_transaction(idx, typology):
    """Generate a suspicious transaction matching a specific AML typology."""
    templates = {
        "structuring": (
            "Cash deposit by {name} into account ending {acct}. Amount: ${amount:,.2f}. "
            "This is the {nth} cash deposit this week, all below $10,000. "
            "Total weekly deposits: ${total:,.2f}. Customer declined to provide "
            "purpose of deposits. Teller notes: customer appeared nervous."
        ),
        "layering_shell_companies": (
            "Wire transfer from {company1} ({country1}) to {company2} ({country2}). "
            "Purpose stated as 'consulting services.' Amount: ${amount:,.2f}. "
            "Originator company registered {months} months ago. No web presence found. "
            "Beneficiary is a holding company with no operating history."
        ),
        "trade_based": (
            "International wire from {company1} ({country1}) to {company2} ({country2}) "
            "for 'import of electronic components.' Amount: ${amount:,.2f}. Invoice "
            "indicates 500 units at ${unit:,.2f} each. Market price for stated goods: "
            "${market:,.2f} per unit. Significant price discrepancy noted."
        ),
        "funnel_accounts": (
            "Multiple incoming wires totaling ${amount:,.2f} from {count} different "
            "originators in {country1} into account of {name}. Funds consolidated "
            "and wired out within 24 hours to {company} in {country2}. Account holder "
            "is a recently arrived student with no declared income."
        ),
        "rapid_movement": (
            "Wire received: ${amount:,.2f} from {company1} ({country1}). "
            "Within 3 hours, ${out_amount:,.2f} wired to {company2} ({country2}). "
            "Account holder {name} has no apparent business relationship with either "
            "party. Account opened {weeks} weeks ago with minimum deposit."
        ),
        "cash_intensive": (
            "Cash deposit of ${amount:,.2f} by {name}, owner of {business}. "
            "Reported monthly revenue: ${revenue:,.2f}. This deposit represents "
            "{pct}% of reported monthly revenue in a single transaction. "
            "Business type: {btype}. Location has minimal foot traffic per site visit."
        ),
        "nominee_accounts": (
            "Account opened by {name1} with {name2} listed as authorized signer. "
            "All transactions conducted by {name2}. Wire of ${amount:,.2f} received "
            "from {company} ({country1}). {name1} has no apparent connection to the "
            "originating entity. Address on file is a mail forwarding service."
        ),
        "digital_currency": (
            "Wire transfer of ${amount:,.2f} from {name} to {exchange} cryptocurrency "
            "exchange. Customer previously received ${in_amount:,.2f} in cash deposits "
            "over past {days} days. No declared cryptocurrency business. Customer stated "
            "purpose as 'personal investment' but could not explain source of cash."
        )
    }

    names = ["Viktor S. Petrov", "Liang Wei Chen", "Abdul R. Hassan",
             "Carlos M. Reyes", "Dmitri Volkov", "Fatima Al-Rashidi"]
    companies = ["Oceanic Trading Ltd", "Pacific Rim Holdings", "Global Bridge Corp",
                 "Zenith Ventures SA", "Atlas Capital Group", "Meridian Offshore LLC"]
    businesses = ["Lucky Star Convenience", "Golden Lotus Restaurant",
                  "Express Auto Wash", "Sunrise Laundromat"]
    exchanges = ["Binance", "Kraken", "OKX", "Bybit"]

    template = templates.get(typology, templates["layering_shell_companies"])

    if typology == "structuring":
        amount = np.random.uniform(8000, 9900)
        total = amount * np.random.randint(3, 7)
    elif typology in ["trade_based", "layering_shell_companies"]:
        amount = np.random.lognormal(mean=12.5, sigma=0.8)
    else:
        amount = np.random.lognormal(mean=11, sigma=1.0)
    amount = min(amount, 5000000)

    orig_country = np.random.choice(HIGH_RISK_COUNTRIES + MEDIUM_RISK_COUNTRIES)
    benef_country = np.random.choice(LOW_RISK_COUNTRIES + MEDIUM_RISK_COUNTRIES)

    narrative = template.format(
        name=np.random.choice(names),
        name1=np.random.choice(names),
        name2=np.random.choice(names),
        company=np.random.choice(companies),
        company1=np.random.choice(companies),
        company2=np.random.choice(companies),
        business=np.random.choice(businesses),
        exchange=np.random.choice(exchanges),
        country1=orig_country,
        country2=benef_country,
        acct=np.random.randint(1000, 9999),
        amount=amount,
        total=amount * np.random.randint(3, 7) if typology == "structuring" else amount,
        nth=np.random.choice(["3rd", "4th", "5th", "6th"]),
        months=np.random.randint(1, 6),
        weeks=np.random.randint(2, 8),
        days=np.random.randint(10, 45),
        count=np.random.randint(4, 12),
        unit=amount / 500 * np.random.uniform(2.5, 4.0),
        market=amount / 500 * np.random.uniform(0.3, 0.6),
        out_amount=amount * np.random.uniform(0.92, 0.98),
        in_amount=amount * np.random.uniform(0.8, 1.2),
        revenue=amount * np.random.uniform(0.3, 0.6),
        pct=int(np.random.uniform(150, 400)),
        btype=np.random.choice(["convenience store", "laundromat", "car wash", "restaurant"])
    )

    rule_triggers = np.random.choice(
        ["RULE-042", "RULE-117", "RULE-203", "RULE-089", "RULE-156", "RULE-301"],
        size=np.random.randint(1, 4), replace=False
    ).tolist()

    return {
        "transaction_id": f"TXN-2024-{idx:07d}",
        "narrative": narrative,
        "amount": round(amount, 2),
        "currency": "USD",
        "originator_country": orig_country,
        "beneficiary_country": benef_country,
        "transaction_type": np.random.choice(["wire", "cash_deposit"], p=[0.6, 0.4]),
        "originator_entity_type": np.random.choice(["corporation", "individual"], p=[0.5, 0.5]),
        "customer_risk_rating": np.random.choice(["high", "medium"], p=[0.7, 0.3]),
        "rule_triggers": rule_triggers,
        "label": 1,
        "typology": typology
    }


# Generate dataset
transactions = []
typology_list = list(AML_TYPOLOGIES.keys())

# 500 suspicious transactions (evenly distributed across typologies)
for i in range(500):
    typology = typology_list[i % len(typology_list)]
    transactions.append(generate_suspicious_transaction(i, typology))

# 4500 benign transactions
for i in range(500, 5000):
    transactions.append(generate_benign_transaction(i))

np.random.shuffle(transactions)
df = pd.DataFrame(transactions)

print(f"Dataset: {len(df)} transactions")
print(f"  Suspicious: {df['label'].sum()} ({df['label'].mean()*100:.1f}%)")
print(f"  Benign: {(1-df['label']).sum():.0f} ({(1-df['label'].mean())*100:.1f}%)")
print(f"\nTypology distribution:")
print(df[df['label']==1]['typology'].value_counts())

## 3. Exploratory Data Analysis

Before building the pipeline, we need to understand the data distributions. This mirrors what a real compliance data scientist would do before deploying any model.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 3.1 Amount distribution by label
ax = axes[0, 0]
benign_amounts = df[df['label']==0]['amount']
suspicious_amounts = df[df['label']==1]['amount']
ax.hist(np.log10(benign_amounts + 1), bins=50, alpha=0.6, label='Benign', color='steelblue')
ax.hist(np.log10(suspicious_amounts + 1), bins=50, alpha=0.6, label='Suspicious', color='crimson')
ax.set_xlabel('log10(Amount)')
ax.set_ylabel('Count')
ax.set_title('Transaction Amount Distribution')
ax.legend()

# 3.2 Country risk distribution
ax = axes[0, 1]
def classify_country(c):
    if c in HIGH_RISK_COUNTRIES: return 'High Risk'
    elif c in MEDIUM_RISK_COUNTRIES: return 'Medium Risk'
    else: return 'Low Risk'

df['orig_risk'] = df['originator_country'].apply(classify_country)
risk_by_label = df.groupby(['orig_risk', 'label']).size().unstack(fill_value=0)
risk_by_label.plot(kind='bar', ax=ax, color=['steelblue', 'crimson'])
ax.set_title('Originator Country Risk by Label')
ax.set_ylabel('Count')
ax.legend(['Benign', 'Suspicious'])
ax.tick_params(axis='x', rotation=0)

# 3.3 Narrative length
ax = axes[1, 0]
df['narrative_len'] = df['narrative'].str.len()
ax.hist(df[df['label']==0]['narrative_len'], bins=40, alpha=0.6,
        label='Benign', color='steelblue')
ax.hist(df[df['label']==1]['narrative_len'], bins=40, alpha=0.6,
        label='Suspicious', color='crimson')
ax.set_xlabel('Narrative Length (characters)')
ax.set_ylabel('Count')
ax.set_title('Narrative Length Distribution')
ax.legend()

# 3.4 Typology distribution
ax = axes[1, 1]
typology_counts = df[df['label']==1]['typology'].value_counts()
ax.barh(typology_counts.index, typology_counts.values, color='crimson', alpha=0.7)
ax.set_xlabel('Count')
ax.set_title('Suspicious Transaction Typologies')

plt.tight_layout()
plt.savefig('eda_overview.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nMedian amount — Benign: ${benign_amounts.median():,.2f}")
print(f"Median amount — Suspicious: ${suspicious_amounts.median():,.2f}")

## 4. TODO: Guided EDA Questions

Answer these questions using the data:

In [ ]:
def eda_analysis(df):
    """
    Analyze the dataset and answer key questions about the data distribution.

    Returns a dictionary with:
    - 'top_3_suspicious_countries': list of 3 most common originator countries
                                     in suspicious transactions
    - 'rule_trigger_precision': dict mapping each rule ID to its precision
                                 (fraction of transactions with that rule that
                                  are truly suspicious)
    - 'median_amount_ratio': ratio of suspicious median amount to benign median
    """
    # ============ TODO ============
    # 1. Find the top 3 originator countries in suspicious transactions
    # 2. For each unique rule trigger, compute precision =
    #    (suspicious with rule) / (total with rule)
    # 3. Compute the ratio of suspicious median amount to benign median amount
    # ==============================

    results = {
        'top_3_suspicious_countries': ???,  # YOUR CODE HERE
        'rule_trigger_precision': ???,       # YOUR CODE HERE
        'median_amount_ratio': ???           # YOUR CODE HERE
    }

    return results

In [ ]:
# Verification
results = eda_analysis(df)
assert len(results['top_3_suspicious_countries']) == 3, "Should return exactly 3 countries"
assert all(isinstance(c, str) for c in results['top_3_suspicious_countries']), "Countries should be strings"
assert isinstance(results['rule_trigger_precision'], dict), "Should be a dict of rule -> precision"
assert results['median_amount_ratio'] > 1.0, "Suspicious amounts should be higher on average"
print("Correct! Your EDA analysis is complete.")
print(f"Top suspicious countries: {results['top_3_suspicious_countries']}")
print(f"Median amount ratio: {results['median_amount_ratio']:.2f}x")

## 5. Baseline 1: Rule-Based Screening

The current system uses keyword and threshold rules. Let us implement a simplified version and measure its performance.

In [ ]:
def rule_based_screen(transaction):
    """
    Simple rule-based AML screening.
    Returns 1 (suspicious) or 0 (benign).
    """
    score = 0

    # Rule: high-risk country
    if transaction['originator_country'] in HIGH_RISK_COUNTRIES:
        score += 3
    elif transaction['originator_country'] in MEDIUM_RISK_COUNTRIES:
        score += 1

    # Rule: large amount
    if transaction['amount'] > 100000:
        score += 2
    elif transaction['amount'] > 50000:
        score += 1

    # Rule: keyword matching in narrative
    suspicious_keywords = ['shell', 'offshore', 'nominee', 'no apparent',
                          'declined to provide', 'nervous', 'discrepancy',
                          'no web presence', 'mail forwarding', 'cryptocurrency']
    narrative_lower = transaction['narrative'].lower()
    keyword_hits = sum(1 for kw in suspicious_keywords if kw in narrative_lower)
    score += keyword_hits * 2

    # Rule: high risk rating
    if transaction['customer_risk_rating'] == 'high':
        score += 2

    return 1 if score >= 3 else 0

# Apply rule-based screening
df['rule_pred'] = df.apply(rule_based_screen, axis=1)

# Evaluate
rule_recall = recall_score(df['label'], df['rule_pred'])
rule_precision = precision_score(df['label'], df['rule_pred'])
rule_fp = df[(df['rule_pred']==1) & (df['label']==0)].shape[0]
rule_tp = df[(df['rule_pred']==1) & (df['label']==1)].shape[0]
rule_fn = df[(df['rule_pred']==0) & (df['label']==1)].shape[0]

print("=== Rule-Based Baseline ===")
print(f"SAR Capture Rate (Recall): {rule_recall:.3f}")
print(f"Precision: {rule_precision:.3f}")
print(f"True Positives: {rule_tp}")
print(f"False Positives: {rule_fp}")
print(f"False Negatives: {rule_fn}")
print(f"False Positive Rate among flagged: {rule_fp/(rule_fp+rule_tp):.3f}")

## 6. Baseline 2: Zero-Shot LLM Screening

Now let us see what happens when we use an LLM with a minimal prompt — no prompt design techniques at all.

In [ ]:
def zero_shot_screen(transaction, client, model):
    """
    Zero-shot LLM screening with no prompt design.
    Returns prediction (0 or 1) and raw response.
    """
    prompt = f"""Is this financial transaction suspicious for money laundering?
Answer with just "yes" or "no".

Transaction: {transaction['narrative']}
Amount: ${transaction['amount']:,.2f}
Originator country: {transaction['originator_country']}

Answer:"""

    response = client.messages.create(
        model=model,
        max_tokens=10,
        messages=[{"role": "user", "content": prompt}]
    )
    answer = response.content[0].text.strip().lower()
    pred = 1 if 'yes' in answer else 0
    return pred, answer


# Run zero-shot on a sample (full dataset would be expensive)
sample_size = 200
sample_df = pd.concat([
    df[df['label']==1].sample(50, random_state=42),
    df[df['label']==0].sample(150, random_state=42)
]).reset_index(drop=True)

print(f"Running zero-shot screening on {sample_size} transactions...")
zero_shot_preds = []
for idx, row in sample_df.iterrows():
    pred, _ = zero_shot_screen(row, client, MODEL)
    zero_shot_preds.append(pred)
    if (idx + 1) % 50 == 0:
        print(f"  Processed {idx+1}/{sample_size}")

sample_df['zs_pred'] = zero_shot_preds

zs_recall = recall_score(sample_df['label'], sample_df['zs_pred'])
zs_precision = precision_score(sample_df['label'], sample_df['zs_pred'])
print(f"\n=== Zero-Shot LLM Baseline ===")
print(f"SAR Capture Rate (Recall): {zs_recall:.3f}")
print(f"Precision: {zs_precision:.3f}")

## 7. Building the Prompt Chain — Step 1: Risk Factor Extraction

Now we apply prompt design principles. Step 1 uses a **system prompt** with **structured output** to extract risk factors from the transaction.

In [ ]:
STEP1_SYSTEM_PROMPT = """You are a senior BSA/AML compliance analyst at a major US financial institution.
You have 15 years of experience reviewing transactions for suspicious activity under
the Bank Secrecy Act, USA PATRIOT Act, and FinCEN regulations.

Your task is to extract risk factors from transaction records. For each transaction,
identify specific indicators that could suggest money laundering, terrorist financing,
sanctions evasion, or fraud.

Focus on:
- Geographic risk (high-risk jurisdictions, tax havens, sanctioned countries)
- Transaction patterns (structuring, rapid movement, round amounts)
- Entity risk (shell companies, nominees, recently formed entities)
- Narrative red flags (vague purpose, inconsistencies, missing information)
- Behavioral indicators (nervousness, refusal to provide information)

Be thorough but precise. Only flag indicators that are genuinely present in the data.
Do not speculate or invent risk factors that are not supported by the transaction record."""


def step1_extract_risk_factors(transaction, client, model):
    """
    Step 1: Extract risk factors using system prompt + structured output.
    """
    user_prompt = f"""Analyze this transaction and extract all risk factors.

Transaction ID: {transaction['transaction_id']}
Narrative: {transaction['narrative']}
Amount: ${transaction['amount']:,.2f}
Currency: {transaction['currency']}
Originator Country: {transaction['originator_country']}
Beneficiary Country: {transaction['beneficiary_country']}
Transaction Type: {transaction['transaction_type']}
Entity Type: {transaction['originator_entity_type']}
Customer Risk Rating: {transaction['customer_risk_rating']}

Return your analysis as JSON with this exact structure:
{{
  "risk_factors": [
    {{"factor": "description of risk factor", "category": "geographic|transactional|entity|behavioral|narrative", "severity": "high|medium|low"}}
  ],
  "overall_risk_indicator_count": <integer>,
  "high_severity_count": <integer>
}}

Return ONLY the JSON object, no other text."""

    response = client.messages.create(
        model=model,
        max_tokens=800,
        system=STEP1_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )
    return response.content[0].text.strip()


# Test on one suspicious and one benign transaction
suspicious_sample = df[df['label']==1].iloc[0]
benign_sample = df[df['label']==0].iloc[0]

print("=== Suspicious Transaction ===")
print(f"Narrative: {suspicious_sample['narrative'][:200]}...")
result_s = step1_extract_risk_factors(suspicious_sample, client, MODEL)
print(f"\nRisk Factors:\n{result_s}")

print("\n" + "="*60)
print("\n=== Benign Transaction ===")
print(f"Narrative: {benign_sample['narrative'][:200]}...")
result_b = step1_extract_risk_factors(benign_sample, client, MODEL)
print(f"\nRisk Factors:\n{result_b}")

## 8. Gate Check 1: Validate Risk Factor Output

Gate checks catch malformed outputs before they corrupt downstream steps. This is essential for pipeline reliability.

In [ ]:
def gate_check_1(step1_output):
    """
    Validate Step 1 output structure.
    Returns (is_valid, parsed_data, error_message).
    """
    try:
        # Handle potential markdown code fences
        text = step1_output.strip()
        if text.startswith("```"):
            text = text.split("\n", 1)[1]
            text = text.rsplit("```", 1)[0]

        data = json.loads(text)

        # Check required fields
        if "risk_factors" not in data:
            return False, None, "Missing 'risk_factors' field"
        if not isinstance(data["risk_factors"], list):
            return False, None, "'risk_factors' must be a list"

        # Validate each risk factor
        valid_categories = {"geographic", "transactional", "entity", "behavioral", "narrative"}
        valid_severities = {"high", "medium", "low"}

        for i, rf in enumerate(data["risk_factors"]):
            if "factor" not in rf:
                return False, None, f"Risk factor {i} missing 'factor' field"
            if "category" not in rf or rf["category"] not in valid_categories:
                return False, None, f"Risk factor {i} has invalid category"
            if "severity" not in rf or rf["severity"] not in valid_severities:
                return False, None, f"Risk factor {i} has invalid severity"

        return True, data, None

    except json.JSONDecodeError as e:
        return False, None, f"JSON parse error: {str(e)}"


# Test gate check
is_valid, parsed, error = gate_check_1(result_s)
print(f"Suspicious transaction gate check: {'PASS' if is_valid else 'FAIL'}")
if error:
    print(f"  Error: {error}")
else:
    print(f"  Risk factors found: {len(parsed['risk_factors'])}")
    print(f"  High severity: {sum(1 for rf in parsed['risk_factors'] if rf['severity']=='high')}")

## 9. Step 2: Typology Matching with Few-Shot Learning

This step uses **few-shot learning** to match extracted risk factors against known AML typologies. The examples calibrate the model's understanding of how risk factors map to specific laundering patterns.

In [ ]:
# Curated few-shot examples for typology matching
FEW_SHOT_TYPOLOGY_EXAMPLES = [
    {
        "risk_factors": [
            {"factor": "Multiple cash deposits below $10,000 in single week", "category": "transactional", "severity": "high"},
            {"factor": "Customer declined to state purpose", "category": "behavioral", "severity": "medium"},
            {"factor": "Total weekly deposits exceed $35,000", "category": "transactional", "severity": "high"}
        ],
        "typology": "structuring",
        "reasoning": "Multiple sub-threshold cash deposits within a short period is the defining pattern of structuring (also called 'smurfing'). The customer's refusal to explain the deposits strengthens this assessment."
    },
    {
        "risk_factors": [
            {"factor": "Originator registered in British Virgin Islands", "category": "geographic", "severity": "high"},
            {"factor": "Company formed 3 months ago", "category": "entity", "severity": "high"},
            {"factor": "No web presence or operating history", "category": "entity", "severity": "medium"},
            {"factor": "Vague transaction purpose: 'consulting services'", "category": "narrative", "severity": "medium"}
        ],
        "typology": "layering_shell_companies",
        "reasoning": "A newly formed BVI entity with no operating history sending funds for vague 'consulting' purposes matches the shell company layering typology. The lack of legitimate business indicators combined with an offshore jurisdiction strongly suggests layering."
    },
    {
        "risk_factors": [
            {"factor": "Regular monthly payment to known vendor", "category": "transactional", "severity": "low"},
            {"factor": "Amount consistent with prior payments", "category": "transactional", "severity": "low"},
            {"factor": "Domestic transaction between US entities", "category": "geographic", "severity": "low"}
        ],
        "typology": "none",
        "reasoning": "This transaction shows all hallmarks of legitimate business activity: regular timing, consistent amounts, known counterparty, and domestic routing. No AML typology indicators are present."
    }
]


def step2_typology_matching(risk_factors_data, client, model):
    """
    Step 2: Match risk factors to AML typologies using few-shot learning.
    """
    # Build few-shot examples
    examples_text = ""
    for ex in FEW_SHOT_TYPOLOGY_EXAMPLES:
        factors_str = json.dumps(ex["risk_factors"], indent=2)
        examples_text += f"""Risk Factors: {factors_str}
Typology: {ex['typology']}
Reasoning: {ex['reasoning']}

---
"""

    user_prompt = f"""Given extracted risk factors from a transaction, identify the most likely
AML typology. Use the examples below as reference for how risk factors map to typologies.

Known typologies: structuring, layering_shell_companies, trade_based, funnel_accounts,
cash_intensive, rapid_movement, nominee_accounts, digital_currency, none

{examples_text}

Now analyze these risk factors:
Risk Factors: {json.dumps(risk_factors_data['risk_factors'], indent=2)}

Return as JSON:
{{
  "typology": "<most likely typology or 'none'>",
  "confidence": <0.0 to 1.0>,
  "reasoning": "<why this typology matches>"
}}

Return ONLY the JSON object."""

    response = client.messages.create(
        model=model,
        max_tokens=500,
        messages=[{"role": "user", "content": user_prompt}]
    )
    return response.content[0].text.strip()


# Test typology matching
if is_valid:
    typology_result = step2_typology_matching(parsed, client, MODEL)
    print("=== Typology Matching Result ===")
    print(typology_result)

## 10. TODO: Implement Step 3 — Risk Assessment with Chain-of-Thought

This is the most critical step. You must design a **chain-of-thought prompt** that forces the model to reason step-by-step through the risk assessment.

In [ ]:
def step3_risk_assessment(risk_factors_data, typology_data, transaction, client, model):
    """
    Step 3: Perform risk assessment using Chain-of-Thought prompting.

    The prompt must force the model to:
    1. List all identified risk factors and their severities
    2. Consider the matched typology and confidence
    3. Evaluate mitigating factors (if any)
    4. Weigh the evidence step by step
    5. Arrive at a risk level: "high", "medium", "low", or "clear"

    Args:
        risk_factors_data: Parsed output from Step 1 (dict with 'risk_factors' list)
        typology_data: Parsed output from Step 2 (dict with 'typology', 'confidence', 'reasoning')
        transaction: Original transaction record (dict)
        client: Anthropic client
        model: Model name

    Returns:
        str: JSON string with risk_level, reasoning_chain, and mitigating_factors
    """
    # ============ TODO ============
    # Design a CoT prompt that includes:
    #
    # 1. A system message establishing the analyst role (reuse STEP1_SYSTEM_PROMPT
    #    or write a specialized one)
    #
    # 2. A user message that:
    #    a. Presents the risk factors from Step 1
    #    b. Presents the typology match from Step 2
    #    c. Includes the original transaction details for context
    #    d. Includes a CoT demonstration showing how to reason through
    #       a risk assessment step by step (similar to the baker example
    #       in the article)
    #    e. Asks for output in this JSON format:
    #       {
    #         "risk_level": "high|medium|low|clear",
    #         "reasoning_chain": "Step 1: ... Step 2: ... Step 3: ...",
    #         "mitigating_factors": ["list of factors that reduce risk"],
    #         "aggravating_factors": ["list of factors that increase risk"],
    #         "confidence": 0.0 to 1.0
    #       }
    #
    # HINTS:
    # - The CoT demonstration should show explicit step-by-step reasoning
    # - Include one example where risk is high AND one where risk is clear
    # - Use "Let me reason through this step by step:" as the trigger phrase
    # - Make the model count high-severity vs low-severity factors
    # - The prompt should reference specific BSA/AML regulations
    # ==============================

    cot_prompt = ???  # YOUR CODE HERE

    response = client.messages.create(
        model=model,
        max_tokens=1000,
        system=STEP1_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": cot_prompt}]
    )
    return response.content[0].text.strip()

In [ ]:
# Verification: test your CoT prompt
if is_valid:
    # Parse typology result
    typology_text = typology_result.strip()
    if typology_text.startswith("```"):
        typology_text = typology_text.split("\n", 1)[1].rsplit("```", 1)[0]
    typology_parsed = json.loads(typology_text)

    risk_result = step3_risk_assessment(parsed, typology_parsed, suspicious_sample, client, MODEL)
    print("=== Risk Assessment (CoT) ===")
    print(risk_result)

    # Validate output
    risk_text = risk_result.strip()
    if risk_text.startswith("```"):
        risk_text = risk_text.split("\n", 1)[1].rsplit("```", 1)[0]
    risk_parsed = json.loads(risk_text)

    assert "risk_level" in risk_parsed, "Missing risk_level"
    assert "reasoning_chain" in risk_parsed, "Missing reasoning_chain"
    assert risk_parsed["risk_level"] in ["high", "medium", "low", "clear"], "Invalid risk level"
    assert len(risk_parsed["reasoning_chain"]) > 100, "Reasoning chain too short — CoT not working"
    print("\nCoT prompt is working correctly!")
    print(f"Risk level: {risk_parsed['risk_level']}")
    print(f"Reasoning length: {len(risk_parsed['reasoning_chain'])} chars")

## 11. Step 4: Decision Synthesis with Structured Output

The final step synthesizes all prior analysis into the structured decision object that feeds into the case management system.

In [ ]:
def step4_decision_synthesis(risk_factors_data, typology_data, risk_assessment_data,
                              transaction, client, model):
    """
    Step 4: Synthesize final decision with structured output.
    """
    user_prompt = f"""Based on the complete analysis below, produce the final screening decision.

Transaction ID: {transaction['transaction_id']}
Amount: ${transaction['amount']:,.2f}
Originator: {transaction['originator_country']}
Type: {transaction['transaction_type']}

Risk Factors Identified: {json.dumps(risk_factors_data.get('risk_factors', []))}
Typology Match: {json.dumps(typology_data)}
Risk Assessment: {json.dumps(risk_assessment_data)}

Produce the final decision as JSON:
{{
  "transaction_id": "{transaction['transaction_id']}",
  "risk_assessment": "<risk level from step 3>",
  "recommendation": "file_sar|escalate_to_senior|clear_with_note|clear",
  "confidence": <0.0 to 1.0>,
  "reasoning": {{
    "risk_factors_identified": [<summary list of key risk factors>],
    "typology_match": "<matched typology or none>",
    "mitigating_factors": [<any mitigating factors>],
    "reasoning_chain": "<brief summary of the reasoning>"
  }},
  "regulatory_citations": [<applicable BSA/AML regulation sections>]
}}

Decision rules:
- risk_level "high" + confidence > 0.8 -> "file_sar"
- risk_level "high" + confidence <= 0.8 -> "escalate_to_senior"
- risk_level "medium" -> "escalate_to_senior"
- risk_level "low" -> "clear_with_note"
- risk_level "clear" -> "clear"

Return ONLY the JSON object."""

    response = client.messages.create(
        model=model,
        max_tokens=800,
        system=STEP1_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": user_prompt}]
    )
    return response.content[0].text.strip()

## 12. Full Pipeline with Gate Checks

Now we assemble the complete 4-step pipeline with gate checks between each step.

In [ ]:
def run_full_pipeline(transaction, client, model, verbose=False):
    """
    Run the complete AML screening pipeline.
    Returns (decision_dict, metadata_dict).
    """
    metadata = {"steps_completed": 0, "gate_failures": 0, "total_tokens": 0}
    start_time = time.time()

    # Step 1: Risk Factor Extraction
    try:
        step1_raw = step1_extract_risk_factors(transaction, client, model)
        is_valid, step1_parsed, error = gate_check_1(step1_raw)
        if not is_valid:
            metadata["gate_failures"] += 1
            if verbose:
                print(f"  Gate 1 FAIL: {error}. Retrying...")
            step1_raw = step1_extract_risk_factors(transaction, client, model)
            is_valid, step1_parsed, error = gate_check_1(step1_raw)
            if not is_valid:
                return {"recommendation": "escalate_to_senior",
                        "error": f"Gate 1 failed: {error}"}, metadata
        metadata["steps_completed"] = 1
    except Exception as e:
        return {"recommendation": "escalate_to_senior",
                "error": f"Step 1 error: {str(e)}"}, metadata

    # Step 2: Typology Matching
    try:
        step2_raw = step2_typology_matching(step1_parsed, client, model)
        step2_text = step2_raw.strip()
        if step2_text.startswith("```"):
            step2_text = step2_text.split("\n", 1)[1].rsplit("```", 1)[0]
        step2_parsed = json.loads(step2_text)
        metadata["steps_completed"] = 2
    except (json.JSONDecodeError, Exception) as e:
        metadata["gate_failures"] += 1
        step2_parsed = {"typology": "unknown", "confidence": 0.5,
                        "reasoning": "Typology matching failed"}
        metadata["steps_completed"] = 2

    # Step 3: Risk Assessment (CoT)
    try:
        step3_raw = step3_risk_assessment(step1_parsed, step2_parsed,
                                           transaction, client, model)
        step3_text = step3_raw.strip()
        if step3_text.startswith("```"):
            step3_text = step3_text.split("\n", 1)[1].rsplit("```", 1)[0]
        step3_parsed = json.loads(step3_text)
        metadata["steps_completed"] = 3
    except (json.JSONDecodeError, Exception) as e:
        metadata["gate_failures"] += 1
        step3_parsed = {"risk_level": "medium", "reasoning_chain": "Assessment failed",
                        "confidence": 0.5}
        metadata["steps_completed"] = 3

    # Step 4: Decision Synthesis
    try:
        step4_raw = step4_decision_synthesis(step1_parsed, step2_parsed,
                                              step3_parsed, transaction, client, model)
        step4_text = step4_raw.strip()
        if step4_text.startswith("```"):
            step4_text = step4_text.split("\n", 1)[1].rsplit("```", 1)[0]
        decision = json.loads(step4_text)
        metadata["steps_completed"] = 4
    except (json.JSONDecodeError, Exception) as e:
        metadata["gate_failures"] += 1
        decision = {"recommendation": "escalate_to_senior",
                    "error": f"Step 4 failed: {str(e)}"}

    metadata["total_latency_ms"] = int((time.time() - start_time) * 1000)

    if verbose:
        print(f"  Pipeline complete: {metadata['steps_completed']}/4 steps, "
              f"{metadata['gate_failures']} gate failures, "
              f"{metadata['total_latency_ms']}ms")

    return decision, metadata


# Run full pipeline on one transaction
print("Running full pipeline on suspicious transaction...")
decision, meta = run_full_pipeline(suspicious_sample, client, MODEL, verbose=True)
print(f"\nDecision: {json.dumps(decision, indent=2)}")

## 13. TODO: Implement Embedding-Based Example Selection

Instead of using the same few-shot examples for every transaction, select the most relevant examples based on semantic similarity.

In [ ]:
def build_example_selector(labeled_transactions, model_name='all-MiniLM-L6-v2'):
    """
    Build an embedding-based example selector for few-shot prompts.

    Args:
        labeled_transactions: list of dicts with 'narrative' and 'label' fields
        model_name: sentence-transformers model to use

    Returns:
        A function select(query_narrative, k=3) that returns the k most
        similar labeled transactions to the query.
    """
    # ============ TODO ============
    # 1. Load the sentence-transformers model
    # 2. Compute embeddings for all labeled transaction narratives
    # 3. Store embeddings and transactions
    # 4. Return a closure that:
    #    a. Embeds the query narrative
    #    b. Computes cosine similarity against all stored embeddings
    #    c. Returns the top-k most similar transactions
    #
    # HINTS:
    # - from sentence_transformers import SentenceTransformer
    # - model.encode(texts) returns numpy array of embeddings
    # - Cosine similarity: dot product of normalized vectors
    # - Include both suspicious AND benign examples in the pool
    #   so the model sees both positive and negative cases
    # ==============================

    def select(query_narrative, k=3):
        # YOUR CODE HERE
        return ???

    return select

In [ ]:
# Verification
from sentence_transformers import SentenceTransformer

# Build selector from a sample of labeled data
example_pool = df.sample(200, random_state=42).to_dict('records')
selector = build_example_selector(example_pool)

# Test: find similar examples for a suspicious transaction
test_txn = df[df['label']==1].iloc[5]
similar = selector(test_txn['narrative'], k=3)

assert len(similar) == 3, "Should return exactly 3 examples"
assert all('narrative' in s for s in similar), "Each example should have a narrative"
assert all('label' in s for s in similar), "Each example should have a label"
print("Example selector working correctly!")
for i, ex in enumerate(similar):
    print(f"\n  Example {i+1} (label={ex['label']}):")
    print(f"    {ex['narrative'][:120]}...")

## 14. Pipeline Evaluation on Test Set

Run the full pipeline on a held-out test set and compare against baselines.

In [ ]:
# Split into eval set
eval_size = 100  # Keep small to manage API costs
eval_df = pd.concat([
    df[df['label']==1].sample(25, random_state=99),
    df[df['label']==0].sample(75, random_state=99)
]).reset_index(drop=True)

print(f"Evaluating pipeline on {eval_size} transactions...")
print(f"  Suspicious: {eval_df['label'].sum()}")
print(f"  Benign: {(1-eval_df['label']).sum():.0f}")

pipeline_results = []
for idx, row in eval_df.iterrows():
    decision, meta = run_full_pipeline(row, client, MODEL)
    recommendation = decision.get('recommendation', 'escalate_to_senior')

    # Map recommendation to binary prediction
    pred = 1 if recommendation in ['file_sar', 'escalate_to_senior'] else 0

    pipeline_results.append({
        'transaction_id': row['transaction_id'],
        'label': row['label'],
        'prediction': pred,
        'recommendation': recommendation,
        'confidence': decision.get('confidence', 0.5),
        'latency_ms': meta.get('total_latency_ms', 0),
        'gate_failures': meta.get('gate_failures', 0)
    })

    if (idx + 1) % 25 == 0:
        print(f"  Processed {idx+1}/{eval_size}")

results_df = pd.DataFrame(pipeline_results)

In [ ]:
# Compute metrics
pipeline_recall = recall_score(results_df['label'], results_df['prediction'])
pipeline_precision = precision_score(results_df['label'], results_df['prediction'])
pipeline_fp = results_df[(results_df['prediction']==1) & (results_df['label']==0)].shape[0]
pipeline_tp = results_df[(results_df['prediction']==1) & (results_df['label']==1)].shape[0]
pipeline_fn = results_df[(results_df['prediction']==0) & (results_df['label']==1)].shape[0]

# Also compute rule baseline on same eval set
eval_df['rule_pred'] = eval_df.apply(rule_based_screen, axis=1)
eval_rule_recall = recall_score(eval_df['label'], eval_df['rule_pred'])
eval_rule_precision = precision_score(eval_df['label'], eval_df['rule_pred'])

print("=" * 50)
print("COMPARISON: Rule-Based vs. Prompt Design Pipeline")
print("=" * 50)
print(f"\n{'Metric':<35} {'Rules':>10} {'Pipeline':>10}")
print("-" * 55)
print(f"{'SAR Capture Rate (Recall)':<35} {eval_rule_recall:>10.3f} {pipeline_recall:>10.3f}")
print(f"{'Precision':<35} {eval_rule_precision:>10.3f} {pipeline_precision:>10.3f}")
print(f"{'Avg Latency (ms)':<35} {'<1':>10} {results_df['latency_ms'].mean():>10.0f}")
print(f"{'Gate Check Failures':<35} {'N/A':>10} {results_df['gate_failures'].sum():>10}")

# FP reduction
rule_fp_eval = eval_df[(eval_df['rule_pred']==1) & (eval_df['label']==0)].shape[0]
fp_reduction = (1 - pipeline_fp / max(rule_fp_eval, 1)) * 100
print(f"\n{'False Positive Reduction':<35} {fp_reduction:>9.1f}%")

In [ ]:
# Visualization: comparison bar chart
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Recall comparison
ax = axes[0]
methods = ['Rule-Based', 'Zero-Shot LLM', 'Prompt Pipeline']
recalls = [eval_rule_recall, zs_recall, pipeline_recall]
colors = ['#c0c0c0', '#ffa07a', '#4caf50']
ax.bar(methods, recalls, color=colors)
ax.set_ylabel('SAR Capture Rate (Recall)')
ax.set_title('Recall Comparison')
ax.set_ylim(0, 1.05)
for i, v in enumerate(recalls):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Precision comparison
ax = axes[1]
precisions = [eval_rule_precision, zs_precision, pipeline_precision]
ax.bar(methods, precisions, color=colors)
ax.set_ylabel('Precision')
ax.set_title('Precision Comparison')
ax.set_ylim(0, 1.05)
for i, v in enumerate(precisions):
    ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

# Latency distribution
ax = axes[2]
ax.hist(results_df['latency_ms'] / 1000, bins=20, color='#4caf50', alpha=0.7)
ax.axvline(x=25, color='red', linestyle='--', label='SLA (25s)')
ax.set_xlabel('Latency (seconds)')
ax.set_ylabel('Count')
ax.set_title('Pipeline Latency Distribution')
ax.legend()

plt.tight_layout()
plt.savefig('pipeline_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## 15. TODO: Error Analysis

Analyze the pipeline's errors to identify prompt design improvements.

In [ ]:
def error_analysis(results_df, eval_df):
    """
    Categorize and analyze pipeline errors.

    Returns a dict with:
    - 'false_negatives': list of transactions the pipeline missed (label=1, pred=0)
    - 'false_positives': list of transactions incorrectly flagged (label=0, pred=1)
    - 'fn_typology_distribution': Counter of typologies among false negatives
    - 'fp_common_patterns': list of common patterns in false positive narratives
    - 'recommendations': list of 3 specific prompt improvements based on the errors

    For each false negative, explain WHY the pipeline might have missed it.
    For each false positive, explain WHAT triggered the false alarm.
    """
    # ============ TODO ============
    # 1. Filter false negatives and false positives from results_df
    # 2. For false negatives, look up the original transaction in eval_df
    #    and analyze which typology was missed
    # 3. For false positives, analyze the narrative to identify what
    #    pattern caused the false alarm
    # 4. Propose 3 specific prompt modifications:
    #    - One for improving the system prompt (Step 1)
    #    - One for improving few-shot examples (Step 2)
    #    - One for improving the CoT reasoning (Step 3)
    # ==============================

    analysis = {
        'false_negatives': ???,        # YOUR CODE HERE
        'false_positives': ???,        # YOUR CODE HERE
        'fn_typology_distribution': ???,  # YOUR CODE HERE
        'fp_common_patterns': ???,     # YOUR CODE HERE
        'recommendations': ???         # YOUR CODE HERE
    }

    return analysis

In [ ]:
# Run error analysis
analysis = error_analysis(results_df, eval_df)

print(f"False Negatives: {len(analysis['false_negatives'])}")
print(f"False Positives: {len(analysis['false_positives'])}")
print(f"\nMissed Typologies: {analysis['fn_typology_distribution']}")
print(f"\nRecommended Prompt Improvements:")
for i, rec in enumerate(analysis['recommendations'], 1):
    print(f"  {i}. {rec}")

## 16. Cost and Throughput Analysis

In [ ]:
# Cost estimation
avg_latency_s = results_df['latency_ms'].mean() / 1000
p95_latency_s = results_df['latency_ms'].quantile(0.95) / 1000

# Estimate token costs (approximate)
avg_tokens_per_txn = 4700  # Based on Section 4 estimates
cost_per_1k_input = 0.003   # Claude Sonnet pricing
cost_per_1k_output = 0.015
avg_input_tokens = 3600
avg_output_tokens = 1100
cost_per_txn = (avg_input_tokens / 1000 * cost_per_1k_input +
                avg_output_tokens / 1000 * cost_per_1k_output)

daily_volume = 42000
daily_cost = daily_volume * cost_per_txn
monthly_cost = daily_cost * 30

# Throughput at different concurrency levels
print("=== Cost and Throughput Analysis ===\n")
print(f"Average latency: {avg_latency_s:.1f}s")
print(f"P95 latency: {p95_latency_s:.1f}s")
print(f"Estimated cost per transaction: ${cost_per_txn:.4f}")
print(f"Daily cost (42K transactions): ${daily_cost:,.2f}")
print(f"Monthly cost: ${monthly_cost:,.2f}")

print(f"\nThroughput at different concurrency levels:")
for concurrency in [1, 5, 10, 20, 50]:
    throughput = concurrency / avg_latency_s * 60  # transactions per minute
    hours_for_daily = daily_volume / (throughput * 60)
    print(f"  Concurrency {concurrency:>3d}: {throughput:>8.1f} txn/min "
          f"({hours_for_daily:.1f}h for daily volume)")

In [ ]:
# Visualize cost-benefit analysis
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Cost comparison
ax = axes[0]
categories = ['Analyst Cost\n(Current)', 'Analyst Cost\n(With Pipeline)', 'Pipeline\nCost']
# False positives * $25/review
current_analyst = 40800 * 25 / 1e6  # millions per day
reduced_analyst = 40800 * 0.4 * 25 / 1e6  # 60% FP reduction
pipeline = daily_cost / 1e6
values = [current_analyst, reduced_analyst, pipeline]
colors_cost = ['#e74c3c', '#f39c12', '#27ae60']
ax.bar(categories, values, color=colors_cost)
ax.set_ylabel('Daily Cost ($ millions)')
ax.set_title('Cost Comparison')
for i, v in enumerate(values):
    ax.text(i, v + 0.01, f'${v:.3f}M', ha='center', fontweight='bold')

# ROI over time
ax = axes[1]
months = np.arange(1, 13)
savings_cumulative = months * (current_analyst - reduced_analyst - pipeline) * 30
setup_cost = 0.15  # $150K setup
net_roi = savings_cumulative - setup_cost
ax.plot(months, net_roi, 'g-o', linewidth=2, label='Net Savings')
ax.axhline(y=0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Months')
ax.set_ylabel('Cumulative Net Savings ($ millions)')
ax.set_title('ROI Over 12 Months')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('cost_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print(f"\nBreak-even: Day 1 (pipeline cost is negligible vs analyst savings)")
print(f"Annual net savings: ${net_roi[-1]:.1f}M")

## 17. TODO: Ethical and Regulatory Assessment

In [ ]:
def ethical_assessment(results_df, eval_df):
    """
    Compute disaggregated fairness metrics and draft regulatory compliance notes.

    Returns a dict with:
    - 'fp_rate_by_country_group': dict mapping country risk group to FP rate
    - 'fp_rate_by_entity_type': dict mapping entity type to FP rate
    - 'disparity_ratio': max FP rate / min FP rate across country groups
    - 'sr_11_7_summary': str with 3-4 sentences addressing OCC SR 11-7
                          Model Risk Management requirements
    """
    # ============ TODO ============
    # 1. Merge results_df with eval_df to get country and entity type info
    # 2. Compute false positive rate for each country risk group
    #    (high-risk, medium-risk, low-risk)
    # 3. Compute false positive rate for each entity type
    #    (corporation vs individual)
    # 4. Calculate the disparity ratio
    # 5. Write a brief SR 11-7 compliance summary addressing:
    #    - Model documentation requirements
    #    - Validation and testing procedures
    #    - Ongoing monitoring plan
    #    - Limitations and assumptions
    # ==============================

    assessment = {
        'fp_rate_by_country_group': ???,  # YOUR CODE HERE
        'fp_rate_by_entity_type': ???,    # YOUR CODE HERE
        'disparity_ratio': ???,           # YOUR CODE HERE
        'sr_11_7_summary': ???            # YOUR CODE HERE
    }

    return assessment

In [ ]:
# Run ethical assessment
assessment = ethical_assessment(results_df, eval_df)

print("=== Fairness Analysis ===")
print(f"\nFP Rate by Country Risk Group:")
for group, rate in assessment['fp_rate_by_country_group'].items():
    print(f"  {group}: {rate:.3f}")
print(f"\nFP Rate by Entity Type:")
for etype, rate in assessment['fp_rate_by_entity_type'].items():
    print(f"  {etype}: {rate:.3f}")
print(f"\nDisparity Ratio: {assessment['disparity_ratio']:.2f}")
print(f"  (>2.0 indicates potential bias concern)")

print(f"\n=== SR 11-7 Compliance Summary ===")
print(assessment['sr_11_7_summary'])

## 18. Summary and Next Steps

In this notebook, you built a complete AML transaction screening pipeline using five prompt design techniques:

1. **System Prompts** — Established the LLM as a domain-expert compliance analyst with specific regulatory knowledge
2. **Few-Shot Learning** — Calibrated typology matching by showing labeled examples of suspicious and benign patterns
3. **Chain-of-Thought** — Forced explicit step-by-step reasoning for auditable risk assessments
4. **Structured Output** — Produced parseable JSON decisions for case management system integration
5. **Prompt Chaining** — Decomposed complex screening into 4 reliable sub-steps with gate checks

You measured the impact of each technique against baselines and demonstrated that thoughtful prompt design can reduce false positives by 60%+ while maintaining high recall — exactly the result Meridian Compliance needs.

For the full production system design, including API specifications, serving infrastructure, monitoring dashboards, A/B testing methodology, and cost analysis, refer to **Section 4** of the case study document.

### Key Takeaways
- Prompt design is not about clever wording — it is about engineering the right information structure
- Gate checks between pipeline steps are essential for production reliability
- Few-shot example selection dramatically affects calibration quality
- Chain-of-thought is critical for auditability in regulated industries
- Cost per transaction is negligible compared to analyst time savings